In [1]:
import random

class DummyLLM:
  def __init__(self):
    print("LLM created")

  def predict(self,prompt):
    response_list = [
        "Delhi is the capital of India",
        "EPL is a football league in England",
        "The capital of England is London",
        "AI stands for Artificial Intelligence",
    ]

    return {'response':random.choice(response_list)}

In [2]:
llm = DummyLLM()
llm

LLM created


In [3]:
llm.predict("What is the capital of India")

{'response': 'AI stands for Artificial Intelligence'}

In [4]:
class DummyPromptTemplate:
  def __init__(self,template,input_variables):
    self.template=template
    self.input_variables=input_variables

  def format(self,input_dict):
    return self.template.format(**input_dict)## string formatting


In [7]:
template = DummyPromptTemplate(
    template="Write a {length} poem about {topic}",
    input_variables=['length','topic']
)


In [9]:
prompt = template.format({"length":"short","topic":"freedom"})

In [10]:
llm = DummyLLM()

LLM created


In [11]:
llm.predict(prompt)

{'response': 'AI stands for Artificial Intelligence'}

In [12]:
class DummyLLMChain:

  def __init__(self,llm,prompt):
    self.llm = llm
    self.prompt = prompt

  def run(self,input_dict):
    final_prompt = self.prompt.format(input_dict)
    result = self.llm.predict(final_prompt)
    return result['response']

In [13]:
template = DummyPromptTemplate(
    template="Write a {length} poem about {topic}",
    input_variables=['length','topic']
)


In [14]:
llm = DummyLLM()


LLM created


In [15]:
chain = DummyLLMChain(llm,template)

In [16]:
chain.run({'length':'short','topic':'freedom'})

'Delhi is the capital of India'

## standardising components

In [19]:
from abc import ABC,abstractmethod


In [20]:
class Runnable(ABC):
  @abstractmethod
  def invoke(input_data):
    pass


In [23]:
## abstraction

import random

class DummyLLM(Runnable):
  def __init__(self):
    print("LLM created")

  def invoke(self,prompt):
    response_list = [
        "Delhi is the capital of India",
        "EPL is a football league in England",
        "The capital of England is London",
        "AI stands for Artificial Intelligence",
    ]
    return {'response':random.choice(response_list)}

  def predict(self,prompt):
    response_list = [
        "Delhi is the capital of India",
        "EPL is a football league in England",
        "The capital of England is London",
        "AI stands for Artificial Intelligence",
    ]

    return {'response':random.choice(response_list)}


In [24]:
llm = DummyLLM()
llm

LLM created


In [25]:
class DummyPromptTemplate(Runnable):
  def __init__(self,template,input_variables):
    self.template=template
    self.input_variables=input_variables

  def invoke(self,input_dict):
    return self.template.format(**input_dict)## string formatting

  def format(self,input_dict):
    return self.template.format(**input_dict)## string formatting


In [36]:
class DummyStrOutputParser(Runnable):

  def __init__(self,):
    pass

  def invoke(self,input_data):
    return input_data['response']


In [27]:
class RunnableConnector(Runnable):

  def __init__(self,runnable_list):
    self.runnable_list=runnable_list

  def invoke(self,input_data):
    for runnable in self.runnable_list:
      input_data = runnable.invoke(input_data)

    return input_data



In [28]:
template = DummyPromptTemplate(
    template="Write a {length} poem about {topic}",
    input_variables=['length','topic']
)


In [29]:
llm = DummyLLM()
llm

LLM created


In [37]:
parser =DummyStrOutputParser()
parser


In [38]:
chain = RunnableConnector([template,llm,parser])

In [39]:
chain.invoke({'length':'short','topic':'freedom'})

'The capital of England is London'

# Combining runnables together to make a bigger chain

In [40]:
template1 = DummyPromptTemplate(
    template="Write a joke about {topic}",
    input_variables=['topic']
)

In [41]:
template2 = DummyPromptTemplate(
    template="Explain the folowing joke {response}",
    input_variables=['response']
)

In [42]:
llm = DummyLLM()
llm

LLM created


In [43]:
parser = DummyStrOutputParser()

In [44]:
chain1 = RunnableConnector([template1,llm])

In [46]:
chain2 = RunnableConnector([template2,llm,parser])

In [47]:
final_chain = RunnableConnector([chain1,chain2])


In [48]:
final_chain.invoke({'topic':'AI'})

'AI stands for Artificial Intelligence'